# Proyecto de Aula Programación en Lenguaje Estadístico

## Entrega 1: diagnóstico inicial en Python

**Integrantes:** Steban Ibarra, Luiyi Brito y Astrid De Angel  
**Asignatura:** Programación en Lenguajes Estadísticos (2026-2)  
**Docente:** Yoselin Nieto G.  
**Fecha:** miércoles 28 de octubre de 2026

Cuaderno de Google Colab para cargar el dataset y revisar tipos, faltantes y duplicados.

## 1. Configuración y carga

Se monta Google Drive o se solicita el archivo, y se cargan los datos en dos versiones: tipada y textual.

In [1]:
# Montaje de Google Drive y búsqueda del archivo.
import sys
from pathlib import Path
import pandas as pd

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")

def buscar_datos():
    rutas = [
        Path("/content/drive/MyDrive/Entrega_1_QBPassers2019/datos/datos_crudos/QBPassers2019.csv"),
        Path("/content/drive/MyDrive/Proyecto-Aula/Entrega_1_QBPassers2019/datos/datos_crudos/QBPassers2019.csv"),
        Path("/content/drive/MyDrive/datos/datos_crudos/QBPassers2019.csv"),
    ]
    for ruta in rutas:
        if ruta.exists():
            return ruta
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        ruta = base / "datos" / "datos_crudos" / "QBPassers2019.csv"
        if ruta.exists():
            return ruta.resolve()
    if EN_COLAB:
        subidos = files.upload()
        if subidos:
            nombre = next(iter(subidos))
            Path(nombre).write_bytes(subidos[nombre])
            return Path(nombre).resolve()
    raise FileNotFoundError("No se encontró QBPassers2019.csv.")

RUTA_DATOS = buscar_datos()
df = pd.read_csv(RUTA_DATOS)
df_texto = pd.read_csv(RUTA_DATOS, dtype=str, keep_default_na=False)

print("Archivo:", RUTA_DATOS.name)
print("Dimensiones:", df.shape)
display(df.head())

Archivo: QBPassers2019.csv
Dimensiones: (50, 15)


,name,firstyear,lastyear,games,attempts,completions,compperc,avercomp,yards,interceptions,interatt,touchdowns,tdsatt,longestPass,rating
0,Aaron Rodgers,2005,NaN,181,6061,3913,64.6,12.0,46946,84,1.39,364,6.01,93,102.4
1,Russell Wilson,2012,NaN,128,3777,2436,64.5,12.2,29734,68,1.80,227,6.01,80,101.2
2,Drew Brees,2001,NaN,275,10161,6867,67.6,11.3,77416,237,2.33,547,5.38,98,98.4
3,Tony Romo,2004,2017.0,150,4335,2829,65.3,12.1,34183,117,2.70,248,5.72,85,97.1
4,Tom Brady,2000,NaN,285,9988,6377,63.8,11.7,74571,179,1.79,541,5.42,99,97.0


## 2. Tipos de datos

Se inspeccionan las dimensiones y el tipo detectado por pandas para cada variable.

In [2]:
print("Información general del DataFrame:")
df.info()

tipos_estadisticos = {
    "name": "Cualitativa nominal",
    "firstyear": "Cuantitativa discreta (año)",
    "lastyear": "Cuantitativa discreta (año)",
    "games": "Cuantitativa discreta",
    "attempts": "Cuantitativa discreta",
    "completions": "Cuantitativa discreta",
    "compperc": "Cuantitativa continua (%)",
    "avercomp": "Cuantitativa continua (yardas)",
    "yards": "Cuantitativa discreta",
    "interceptions": "Cuantitativa discreta",
    "interatt": "Cuantitativa continua (por 100 intentos)",
    "touchdowns": "Cuantitativa discreta",
    "tdsatt": "Cuantitativa continua (por 100 intentos)",
    "longestPass": "Cuantitativa discreta (yardas)",
    "rating": "Cuantitativa continua (índice)",
}

tipos = pd.DataFrame({
    "variable": df.columns,
    "tipo_pandas": [str(t) for t in df.dtypes],
    "tipo_estadistico": [tipos_estadisticos[c] for c in df.columns],
    "n_unicos": [df[c].nunique(dropna=True) for c in df.columns],
    "n_faltantes": [int(df[c].isna().sum()) for c in df.columns],
})

display(tipos)

Información general del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           50 non-null     str    
 1   firstyear      50 non-null     int64  
 2   lastyear       27 non-null     float64
 3   games          50 non-null     int64  
 4   attempts       50 non-null     int64  
 5   completions    50 non-null     int64  
 6   compperc       50 non-null     float64
 7   avercomp       50 non-null     float64
 8   yards          50 non-null     int64  
 9   interceptions  50 non-null     int64  
 10  interatt       50 non-null     float64
 11  touchdowns     50 non-null     int64  
 12  tdsatt         50 non-null     float64
 13  longestPass    50 non-null     int64  
 14  rating         50 non-null     float64
dtypes: float64(6), int64(8), str(1)
memory usage: 6.0 KB


,variable,tipo_pandas,tipo_estadistico,n_unicos,n_faltantes
0,name,str,Cualitativa nominal,50,0
1,firstyear,int64,Cuantitativa discreta (año),26,0
2,lastyear,float64,Cuantitativa discreta (año),17,23
3,games,int64,Cuantitativa discreta,46,0
4,attempts,int64,Cuantitativa discreta,50,0
5,completions,int64,Cuantitativa discreta,50,0
6,compperc,float64,Cuantitativa continua (%),38,0
7,avercomp,float64,Cuantitativa continua (yardas),20,0
8,yards,int64,Cuantitativa discreta,50,0
9,interceptions,int64,Cuantitativa discreta,46,0


## 3. Valores faltantes

Se cuentan los faltantes por variable y su porcentaje.

In [3]:
faltantes = pd.DataFrame({
    "n_faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2),
}).sort_values("n_faltantes", ascending=False)

display(faltantes[faltantes["n_faltantes"] > 0])
print(
    f"Total de celdas faltantes: {int(df.isna().sum().sum())} "
    f"({df.isna().mean().mean() * 100:.2f}% de la matriz)."
)

,n_faltantes,porcentaje
lastyear,23,46.0


Total de celdas faltantes: 23 (3.07% de la matriz).


## 4. Registros duplicados

Se verifican filas duplicadas y nombres de jugador repetidos.

In [4]:
duplicados = pd.DataFrame({
    "verificacion": [
        "Filas completamente duplicadas",
        "Nombres de jugador duplicados",
        "Nombres únicos",
    ],
    "n": [
        int(df.duplicated().sum()),
        int(df.duplicated(subset=["name"]).sum()),
        int(df["name"].nunique()),
    ],
})

display(duplicados)

,verificacion,n
0,Filas completamente duplicadas,0
1,Nombres de jugador duplicados,0
2,Nombres únicos,50


## 5. Consistencia y formato

Se recalculan las tasas y el rating, y se revisan los ceros iniciales de `tdsatt` e `interatt`.

In [5]:
# Conversión defensiva y recálculo de tasas.
df["lastyear"] = df["lastyear"].astype("Int64")
df["tdsatt"] = pd.to_numeric(df["tdsatt"], errors="coerce")
df["interatt"] = pd.to_numeric(df["interatt"], errors="coerce")

df["comp_calc"] = df["completions"] / df["attempts"] * 100
df["avercomp_calc"] = df["yards"] / df["completions"]
df["tdsatt_calc"] = df["touchdowns"] / df["attempts"] * 100
df["interatt_calc"] = df["interceptions"] / df["attempts"] * 100

def limitar(valor):
    return valor.clip(lower=0, upper=2.375)

a = limitar((df["completions"] / df["attempts"] - 0.30) * 5)
b = limitar((df["yards"] / df["attempts"] - 3.00) * 0.25)
c = limitar((df["touchdowns"] / df["attempts"]) * 20)
d = limitar(2.375 - (df["interceptions"] / df["attempts"]) * 25)
df["rating_calc"] = ((a + b + c + d) / 6) * 100

def diferencia_maxima(real, calculado):
    return float((real - calculado).abs().max())

consistencia = pd.DataFrame({
    "verificacion": [
        "compperc vs. completions/attempts×100",
        "avercomp vs. yards/completions",
        "tdsatt vs. touchdowns/attempts×100",
        "interatt vs. interceptions/attempts×100",
        "rating vs. fórmula oficial NFL",
    ],
    "diferencia_maxima": [
        diferencia_maxima(df["compperc"], df["comp_calc"]),
        diferencia_maxima(df["avercomp"], df["avercomp_calc"]),
        diferencia_maxima(df["tdsatt"], df["tdsatt_calc"]),
        diferencia_maxima(df["interatt"], df["interatt_calc"]),
        diferencia_maxima(df["rating"], df["rating_calc"]),
    ],
})

ceros = pd.Series({
    "tdsatt": int(df_texto["tdsatt"].str.startswith("0").sum()),
    "interatt": int(df_texto["interatt"].str.startswith("0").sum()),
}, name="ceros_iniciales")

display(consistencia)
display(ceros.to_frame())

,verificacion,diferencia_maxima
0,compperc vs. completions/attempts×100,0.048780
1,avercomp vs. yards/completions,0.049234
2,tdsatt vs. touchdowns/attempts×100,0.004983
3,interatt vs. interceptions/attempts×100,0.004963
4,rating vs. fórmula oficial NFL,0.046359


,ceros_iniciales
tdsatt,6
interatt,6


## 6. Resultados

Se presenta un resumen numérico del diagnóstico.

In [6]:
resumen = pd.DataFrame({
    "indicador": [
        "Filas",
        "Columnas",
        "Filas duplicadas",
        "Nombres duplicados",
        "Celdas faltantes",
        "Faltantes en lastyear",
    ],
    "valor": [
        df.shape[0],
        df.shape[1],
        int(df.duplicated().sum()),
        int(df.duplicated(subset=["name"]).sum()),
        int(df.isna().sum().sum()),
        int(df["lastyear"].isna().sum()),
    ],
})

display(resumen)

,indicador,valor
0,Filas,50
1,Columnas,20
2,Filas duplicadas,0
3,Nombres duplicados,0
4,Celdas faltantes,23
5,Faltantes en lastyear,23
